# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
! pip install -q schedule pytest # установка библиотек, если ещё не


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [91]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup
import re

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [189]:
def get_book_data(book_url: str) -> dict:
    """
    Собирает данные о книге со страницы каталога Books to Scrape.

    Функция загружает страницу книги, разбирает HTML и извлекает название,
    цену, рейтинг, количество в наличии, описание и все характеристики
    из таблицы Product Information.

    Args:
        book_url (str): Ссылка на страницу книги, например
            'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'.

    Returns:
        dict: Словарь с данными о книге:
            - "Название" (str): название книги;
            - "Цена" (float): цена в фунтах;
            - "Рейтинг" (int): рейтинг от 0 до 5;
            - "Количество в наличии" (int): число экземпляров на складе;
            - "Описание" (str | None): описание книги, если оно есть;
            - "Дополнительные характеристики" (dict): строки таблицы
              Product Information в виде «название: значение».
            Если сервер вернул код ответа, отличный от 200, возвращается
            пустой словарь.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    rating = {"Zero": 0, "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    response = requests.get(book_url, timeout=15)
    response.encoding = "utf-8"
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        book_info = dict()
        html_text1 = soup.find(attrs={"class": "product_main"})
        book_name = html_text1.find("h1").get_text()
        book_price = html_text1.find(attrs={"class": "price_color"}).get_text()
        book_rate = html_text1.find(attrs={"class": "star-rating"})['class'][1]
        book_count = html_text1.find(attrs={"class": "instock availability"}).get_text().strip()
        description_header = soup.find(id="product_description")
        book_description = None
        if description_header is not None:
            book_description = description_header.find_next_sibling("p").get_text(strip=True)
        html2_text = soup.find(attrs={"class": "table table-striped"})
        book_info["Название"] = book_name
        book_info["Цена"] = float(re.search(r"\d+(?:\.\d+)?", book_price).group())
        book_info["Рейтинг"] = rating[book_rate]
        book_info["Количество в наличии"] = int(re.search(r'\d+', book_count).group(0))
        book_info["Описание"] = book_description
        add_spec = dict()
        for row in html2_text.find_all("tr"):
            name = row.find("th").get_text()
            value = row.find("td").get_text()

            if name == "UPC":
                add_spec["Универсальный код продукта"] = value
            elif name == "Product Type":
                add_spec["Тип продукта"] = value
            elif name == "Price (excl. tax)":
                add_spec["Цена без налога"] = float(re.search(r"\d+(?:\.\d+)?", value).group())
            elif name == "Price (incl. tax)":
                add_spec["Цена с налогом"] = float(re.search(r"\d+(?:\.\d+)?", value).group())
            elif name == "Tax":
                add_spec["Налог"] = float(re.search(r"\d+(?:\.\d+)?", value).group())
            elif name == "Availability":
                add_spec["Количество в наличии"] = int(re.search(r'\d+', value).group(0))
            elif name == "Number of reviews":
                add_spec["Количество отзывов"] = value
            
        book_info["Дополнительные характеристики"] = add_spec
        return book_info
    else:
        return(dict())
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [190]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'Название': 'A Light in the Attic',
 'Цена': 51.77,
 'Рейтинг': 3,
 'Количество в наличии': 22,
 'Описание': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who pu

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
def scrape_books(is_save=False):
    """
    Собирает данные обо всех книгах каталога Books to Scrape.

    Функция последовательно обходит страницы каталога, собирает ссылки
    на книги, для каждой вызывает get_book_data и при необходимости
    сохраняет результат в файл.

    Args:
        is_save (bool): Если True, результат сохраняется в файл
            books_data.txt: по одному словарю с данными о книге
            на каждой строке. По умолчанию False.

    Returns:
        str: строка, которая содержит данные о книгах (формат описан
            в get_book_data). Страницы каталога, которые не удалось
            загрузить, пропускаются.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    max_page = 50
    books = ""

    for page_number in range(1, max_page + 1):
        catalog_url = f"http://books.toscrape.com/catalogue/page-{page_number}.html"
        response = requests.get(catalog_url, timeout=10)
        response.encoding = "utf-8"
        if response.status_code != 200:
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        for article in soup.find_all("article", class_="product_pod"):
            link = article.find("h3").find("a")["href"]
            book_data = get_book_data("http://books.toscrape.com/catalogue/" + link)
            if book_data:
                books += str(book_data) + '\n'

    if is_save:
        with open("books_data.txt", "w", encoding="utf-8") as f:
            f.write(books)

    return books
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [266]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

<class 'str'> 1792061


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [272]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
schedule.every().day.at("19:00").do(scrape_books, is_save=True)

while True:
    schedule.run_pending()
    time.sleep(30)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [274]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest tests/test_scraper.py

============================= test session starts ==============================
platform darwin -- Python 3.11.1, pytest-9.2.0, pluggy-1.7.0
rootdir: /Users/ruslan/Desktop/Магистратура
plugins: anyio-4.16.0
collected 0 items                                                              

============================ no tests ran in 0.01s =============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Было задано написать парсер веб-сайта с информацией по книгам. Основные задания были выполнены`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Много пришлось обращаться к лекциям, сторонним сайтам, ИИ за помощью в пойске нужного значения в html странице`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: обращался за поиском ошибок в коде с промтом: найди ошибку в коде + [код]. Как найти строчку "название поля" в html [кусок кода html] c помощью bs4`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: написание тесткейсов`

**5. Самооценка по критериям задания**:

`Ответ: Нужно было больше выделить времени на рещение ДЗ`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Было ценно попрактиковатья в парсинге сайта`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

